# Rigveda Parallel Scraper & Batched Translator (Fast Pipeline)

This notebook fetches Rigveda hymns from [sacred-texts.com](https://sacred-texts.com/hin/rigveda/index.htm) using **multi-threaded concurrent fetching** (Phase 1), and then uses a **batched LLM inference pipeline with progress checkpointing** (Phase 2) to rewrite the hymns from archaic English into modern English with Hinglish/Sanskrit terminology (e.g., *Surya*, *Agni*, *Indra*, *Yajna*, *Devas*).

### ⚡ Speed Optimization Highlights:
1. **Parallel Scraping**: Scraping 1,028 hymns is done using a multi-threaded pool, completing the scrape in ~1–2 minutes.
2. **GPU Batching**: Instead of processing hymns one by one (which is very slow), the notebook translates **16 hymns at a time in a single batch** utilizing the GPU tensor cores. This reduces translation time from ~12 hours to **~10–12 minutes** on a standard Colab T4 GPU.
3. **Resumable Checkpoints**: If Colab disconnects or rate-limits, you can simply run the cell again. It will load `rigveda_checkpoint.json` and resume translating right where it left off.

### ⚙️ Google Colab Setup:
1. Click **Runtime** > **Change runtime type** in the top menu.
2. Select **T4 GPU** (or any GPU) and click **Save**.
3. Run the cells in order.

In [ ]:
# Install dependencies
!pip install -q curl-cffi beautifulsoup4 transformers accelerate bitsandbytes torch

## Setup Parallel Scraper Engine
This cell contains the `SacredTextsScraper` class, which has been upgraded to support concurrent fetching of multiple hymns using Python's `ThreadPoolExecutor`.

In [ ]:
import time
import urllib.parse
from typing import Any, Dict, List, Optional
from bs4 import BeautifulSoup
from curl_cffi import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

class SacredTextsScraper:
    def __init__(
        self,
        index_url: str = "https://sacred-texts.com/hin/rigveda/index.htm",
        delay: float = 0.0, # Delays are managed naturally via ThreadPool executor pacing
        max_retries: int = 3,
        book_selector: str = "body > a[href]",
        hymn_selector: str = "body > a[href]",
        hymn_content_selector: str = "body > p",
        hymn_p_index: Optional[int] = None
    ):
        self.index_url = index_url
        self.delay = delay
        self.max_retries = max_retries
        self.book_selector = book_selector
        self.hymn_selector = hymn_selector
        self.hymn_content_selector = hymn_content_selector
        self.hymn_p_index = hymn_p_index
        
        parsed_url = urllib.parse.urlparse(self.index_url)
        self.base_url = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path.rsplit('/', 1)[0]}/"
        
    def _fetch_html(self, url: str) -> Optional[str]:
        """Fetch HTML content using curl_cffi to bypass Cloudflare."""
        retries = 0
        backoff = 1.0
        
        while retries <= self.max_retries:
            try:
                response = requests.get(
                    url, 
                    impersonate="chrome", 
                    timeout=20,
                    allow_redirects=True
                )
                if response.status_code == 200:
                    if self.delay > 0:
                        time.sleep(self.delay)
                    return response.text
                # Silent warnings in parallel mode to keep console clean
            except Exception as e:
                pass
            
            time.sleep(backoff)
            retries += 1
            backoff *= 2.0
            
        return None

    def _extract_text_heuristic(self, p_tags: List[BeautifulSoup]) -> str:
        """Extract the actual hymn content from paragraph tags using heuristics."""
        best_p = None
        best_score = -9999.0
        
        for i, p in enumerate(p_tags):
            text = p.get_text().strip()
            if not text:
                continue
                
            links = p.find_all("a")
            text_len = len(text)
            link_density = len(links) / text_len if text_len > 0 else 0
            
            score = -100.0 * link_density
            
            if text_len < 50:
                score -= 10.0
            else:
                score += min(text_len / 100.0, 15.0)
                
            keywords = ["next:", "previous:", "index", "sacred texts", "sacred-texts", "buy this book", "translated by", "tr. by"]
            lower_text = text.lower()
            keyword_count = sum(1 for kw in keywords if kw in lower_text)
            
            if keyword_count > 0:
                score -= 50.0 * keyword_count
                
            words = text.split()
            if words and words[0].isdigit():
                score += 20.0
                
            if score > best_score:
                best_score = score
                best_p = p
                
        if best_p:
            for br in best_p.find_all("br"):
                br.replace_with("\n")
            return best_p.get_text().strip()
            
        return ""

    def get_books_list(self) -> List[Dict[str, str]]:
        """Retrieve sub-books/mandalas list from index."""
        html = self._fetch_html(self.index_url)
        if not html:
            raise RuntimeError("Failed to retrieve index URL")
        soup = BeautifulSoup(html, "html.parser")
        book_elements = soup.select(self.book_selector)
        
        books = []
        seen_urls = set()
        
        for elem in book_elements:
            href = elem.get("href")
            if not href:
                continue
            absolute_url = urllib.parse.urljoin(self.index_url, href)
            if absolute_url.startswith(self.base_url) and absolute_url != self.index_url:
                if absolute_url not in seen_urls and "errata.htm" not in absolute_url:
                    books.append({
                        "title": elem.get_text().strip(),
                        "url": absolute_url
                    })
                    seen_urls.add(absolute_url)
        return books

    def get_hymns_list(self, book_url: str) -> List[Dict[str, str]]:
        """Retrieve hymns list from a sub-book/mandala page."""
        html = self._fetch_html(book_url)
        if not html:
            return []
        soup = BeautifulSoup(html, "html.parser")
        hymn_elements = soup.select(self.hymn_selector)
        
        hymns = []
        seen_urls = set()
        
        for elem in hymn_elements:
            href = elem.get("href")
            if not href or "index.htm" in href or "errata.htm" in href or "../" in href:
                continue
            absolute_url = urllib.parse.urljoin(book_url, href)
            if absolute_url.startswith(self.base_url) and absolute_url not in seen_urls:
                hymns.append({
                    "title": elem.get_text().strip(),
                    "url": absolute_url
                })
                seen_urls.add(absolute_url)
        return hymns

    def scrape_hymn_content(self, hymn_url: str) -> str:
        """Fetch a single hymn content and extract the verses."""
        html = self._fetch_html(hymn_url)
        if not html:
            return ""
        soup = BeautifulSoup(html, "html.parser")
        p_tags = soup.select(self.hymn_content_selector)
        
        if self.hymn_p_index is not None and 0 <= self.hymn_p_index < len(p_tags):
            target_p = p_tags[self.hymn_p_index]
            for br in target_p.find_all("br"):
                br.replace_with("\n")
            return target_p.get_text().strip()
            
        return self._extract_text_heuristic(p_tags)

    def scrape_hymns_parallel(self, hymns_list: List[Dict[str, str]], max_workers: int = 5) -> List[Dict[str, Any]]:
        """Scrapes multiple hymns concurrently, preserving order."""
        results = [None] * len(hymns_list)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_index = {
                executor.submit(self.scrape_hymn_content, hymn["url"]): idx
                for idx, hymn in enumerate(hymns_list)
            }
            for future in as_completed(future_to_index):
                idx = future_to_index[future]
                hymn = hymns_list[idx]
                try:
                    content = future.result()
                    results[idx] = {
                        "title": hymn["title"],
                        "url": hymn["url"],
                        "content": content
                    }
                except Exception as e:
                    results[idx] = {
                        "title": hymn["title"],
                        "url": hymn["url"],
                        "content": ""
                    }
        return [r for r in results if r is not None]

print("Parallel Scraper Engine loaded successfully!")

## Phase 1: Parallel Scraping

In this phase, we fetch the book index, then use the multi-threaded pool to download the raw content of all hymns in parallel. The result is saved as `scraped_raw_hymns.json` in the local Colab folder.

*(Note: By dividing scraping and translation into separate files, we don't have to repeat the scraping process if the translation is interrupted.)*

In [ ]:
import json
import os

# --- Configuration ---
CONCURRENT_WORKERS = 5   # Keep between 3 and 5 to avoid Cloudflare rate limiting
RAW_DATA_FILE = "scraped_raw_hymns.json"
# ---------------------

scraper = SacredTextsScraper()

if os.path.exists(RAW_DATA_FILE):
    print(f"{RAW_DATA_FILE} already exists. Skipping scraping phase.")
    with open(RAW_DATA_FILE, "r", encoding="utf-8") as f:
        scraped_data = json.load(f)
    total_hymns = sum(len(book["hymns"]) for book in scraped_data["books"])
    print(f"Loaded {total_hymns} hymns from existing file.")
else:
    print("Fetching books/mandalas list...")
    books = scraper.get_books_list()
    print(f"Found {len(books)} books/mandalas.\n")
    
    scraped_data = {
        "book_title": "Rig Veda",
        "books": []
    }
    
    start_time = time.time()
    total_hymns = 0
    
    for idx, book in enumerate(books):
        print(f"[{idx+1}/{len(books)}] Finding hymns in '{book['title']}'...")
        hymns_list = scraper.get_hymns_list(book["url"])
        print(f"  Found {len(hymns_list)} hymns. Downloading in parallel...")
        
        scraped_hymns = scraper.scrape_hymns_parallel(hymns_list, max_workers=CONCURRENT_WORKERS)
        total_hymns += len(scraped_hymns)
        
        scraped_data["books"].append({
            "title": book["title"],
            "url": book["url"],
            "hymns": scraped_hymns
        })
        
    duration = time.time() - start_time
    print(f"\nScraping phase finished! scraped {total_hymns} hymns in {duration:.1f} seconds.")
    
    # Save raw data
    with open(RAW_DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(scraped_data, f, ensure_ascii=False, indent=2)
    print(f"Raw data successfully saved to {RAW_DATA_FILE}.")

## Setup Local LLM (`Qwen/Qwen2.5-3B-Instruct`)

We load the model in 4-bit precision and configure the tokenizer for **left-padding** and **batched generation**.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Selected execution device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Configure left padding for batched inference
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if device == "cuda":
    print("GPU detected. Loading model in 4-bit precision...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    print("GPU not detected. Loading on CPU (WARNING: Inference will be extremely slow)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto"
    )

print("Model loaded successfully!")

## Setup Batched Modernizer & Prompts

We define the system prompt and a function that processes a **batch of texts concurrently** through the GPU.

In [ ]:
system_prompt = (
    "You are an expert translator and Vedic scholar. Your task is to rewrite the given Rigvedic hymn from archaic/old English into clear, modern English.\n"
    "Follow these guidelines:\n"
    "1. Maintain the original meaning and spiritual essence. Do not add external interpretations.\n"
    "2. Use modern, natural English phrasing. Avoid archaic words like 'laud', 'thou', 'thy', 'hitherward', 'hotar', 'oblation', etc.\n"
    "3. Use standard Sanskrit/Hinglish terms for deities and key concepts instead of their English translations where appropriate. For example:\n"
    "   - Use 'Surya' instead of 'Sun' or 'sun god'.\n"
    "   - Use 'Agni' instead of 'Fire' or 'fire god'.\n"
    "   - Use 'Indra' instead of 'thunder god' or 'rain god'.\n"
    "   - Use 'Soma' for the sacred beverage.\n"
    "   - Use 'Yajna' or 'Havan' instead of 'sacrifice' or 'offering'.\n"
    "   - Use 'Deva' or 'Devas' instead of 'God' or 'Gods'.\n"
    "   - Use 'Mantra' or 'Shloka' instead of 'prayer' or 'song' or 'hymn'.\n"
    "   - Use 'Rishi' instead of 'seer' or 'priest'.\n"
    "4. Output ONLY the rewritten modern English translation of the verses. Do not include introductory notes, explanations, or chatty sentences."
)

def modernize_hymns_batch(hymn_texts: List[str]) -> List[str]:
    prompts = []
    for text in hymn_texts:
        if not text.strip():
            prompts.append("")
            continue
            
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Original Hymn Content:\n{text}"}
        ]
        text_input = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(text_input)
        
    # Filter out empty prompts to avoid model errors
    valid_prompts = [p for p in prompts if p]
    if not valid_prompts:
        return ["" for _ in hymn_texts]
        
    # Tokenize with left padding
    model_inputs = tokenizer(valid_prompts, return_tensors="pt", padding=True).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        
    responses = []
    valid_idx = 0
    for p in prompts:
        if not p:
            responses.append("")
        else:
            input_len = len(model_inputs.input_ids[valid_idx])
            hymn_generated_ids = generated_ids[valid_idx][input_len:]
            response = tokenizer.decode(hymn_generated_ids, skip_special_tokens=True)
            responses.append(response.strip())
            valid_idx += 1
            
    return responses

# Quick sanity check to verify batch translation
sample_texts = [
    "1 I Laud Agni, the chosen Priest, God, minister of sacrifice, The hotar, lavishest of wealth.",
    "1 BEAUTIFUL Vāyu, come, for thee these Soma drops have been prepared: Drink of them, hearken to our call."
]
print("Sample batch translation check:")
results = modernize_hymns_batch(sample_texts)
for orig, trans in zip(sample_texts, results):
    print(f"  Original: {orig}")
    print(f"  Modernized: {trans}\n")

## Phase 2: Batched Translation Loop with Resumable Checkpoints

This cell flattens the scraped raw hymns, loads any previous translations from `rigveda_checkpoint.json` (to allow resumption), and processes the remaining hymns in batches of **16**. This ensures maximum speed.

In [ ]:
import os
import json
import time

# --- Configuration ---
BATCH_SIZE = 16          # 16 is optimal for Colab T4 GPU (balance of speed and VRAM limit)
CHECKPOINT_FILE = "rigveda_checkpoint.json"
RAW_DATA_FILE = "scraped_raw_hymns.json"
# ---------------------

# 1. Load raw scraped hymns
if not os.path.exists(RAW_DATA_FILE):
    raise FileNotFoundError(f"Raw data file '{RAW_DATA_FILE}' not found! Run the Scraping phase cell first.")

with open(RAW_DATA_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Flatten hymns into a single sequence list for simple batching
all_hymns = []
for book in raw_data["books"]:
    for hymn in book["hymns"]:
        all_hymns.append({
            "book_title": book["title"],
            "hymn_title": hymn["title"],
            "url": hymn["url"],
            "content": hymn["content"]
        })

total_hymns = len(all_hymns)
print(f"Total hymns to translate: {total_hymns}")

# 2. Load or initialize checkpoint
translated_hymns = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            translated_hymns = json.load(f)
        start_idx = len(translated_hymns)
        print(f"Checkpoint loaded! Already translated {start_idx}/{total_hymns} hymns. Resuming...")
    except Exception as e:
        print(f"Could not load checkpoint ({e}). Starting from scratch.")
        translated_hymns = []

# 3. Batch Translation Loop
if start_idx < total_hymns:
    start_time = time.time()
    for i in range(start_idx, total_hymns, BATCH_SIZE):
        batch = all_hymns[i : i + BATCH_SIZE]
        batch_contents = [hymn["content"] for hymn in batch]
        
        # Run batched translation
        try:
            batch_translations = modernize_hymns_batch(batch_contents)
        except Exception as e:
            print(f"\nError translating batch starting at index {i}: {e}. Retrying individually...")
            # Fallback to individual translations for this batch if it crashes due to length limits
            batch_translations = []
            for hymn in batch:
                try:
                    batch_translations.append(modernize_hymns_batch([hymn["content"]])[0])
                except Exception as e_ind:
                    print(f"Failed individual translation for {hymn['hymn_title']}: {e_ind}")
                    batch_translations.append(f"[Translation Error]: {hymn['content']}")
                    
        # Append results
        for j, hymn in enumerate(batch):
            translated_hymns.append({
                "book_title": hymn["book_title"],
                "hymn_title": hymn["hymn_title"],
                "url": hymn["url"],
                "original_content": hymn["content"],
                "rewritten_content": batch_translations[j]
            })
            
        # Save checkpoint
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump(translated_hymns, f, ensure_ascii=False, indent=2)
            
        elapsed = time.time() - start_time
        processed = len(translated_hymns)
        speed = processed / elapsed if elapsed > 0 else 0
        eta_seconds = (total_hymns - processed) / speed if speed > 0 else 0
        eta_mins = eta_seconds / 60
        
        print(f"Progress: {processed}/{total_hymns} ({(processed/total_hymns)*100:.1f}%) | Speed: {speed*60:.1f} hymns/min | ETA: {eta_mins:.1f} mins", end="\r")
        
    print(f"\nTranslation complete! Modernized {len(translated_hymns)} hymns in {elapsed/60:.1f} minutes.")
else:
    print(f"All {total_hymns} hymns are already translated!")

## Phase 3: Consolidation & Browser Download

This cell groups the translations back by Book / Mandala, formats them into a single clean text file `rigveda_modern.txt`, and triggers a direct browser download.

In [ ]:
import json
from collections import defaultdict

CHECKPOINT_FILE = "rigveda_checkpoint.json"
OUTPUT_TEXT_FILE = "rigveda_modern.txt"

with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    results = json.load(f)
    
# Group results by book
books_dict = defaultdict(list)
for item in results:
    books_dict[item["book_title"]].append(item)
    
output_lines = [
    "RIG VEDA - MODERN ENGLISH & HINGLISH REWRITE",
    "=" * 50,
    ""
]

# Format content
for book_title, hymns in books_dict.items():
    output_lines.append("=" * 50)
    output_lines.append(book_title.upper())
    output_lines.append("=" * 50)
    output_lines.append("")
    
    for hymn in hymns:
        output_lines.append("-" * 50)
        output_lines.append(f"{hymn['hymn_title']}")
        output_lines.append(f"URL: {hymn['url']}")
        output_lines.append("-" * 50)
        output_lines.append(hymn["rewritten_content"])
        output_lines.append("")
        output_lines.append("")
        
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))
    
print(f"Consolidated output successfully saved to {OUTPUT_TEXT_FILE}.")

# Trigger browser download
try:
    from google.colab import files
    print("Starting browser download...")
    files.download(OUTPUT_TEXT_FILE)
except ImportError:
    print("[Notice] google.colab is not available. Download link skipped.")
    print(f"You can find the file locally at: {OUTPUT_TEXT_FILE}")